# T5 Zero-Shot PDE Dialect Evaluation
Minimal notebook: upload dataset.jsonl, run pretrained t5-small (no fine-tune), and output Family Acc / Operator F1 / Structure Acc by dialect.

In [1]:
import sys
!{sys.executable} -m pip install -q transformers torch pandas tqdm

In [2]:
import json, random, re, math
from collections import defaultdict
import numpy as np, pandas as pd, torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
SEED = 42
DIALECTS = ["natural", "latex", "prefix", "postfix"]
CONFIG = {"model_name": "t5-small", "max_input_length": 192, "max_new_tokens": 72, "batch_size": 8}
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
try:
    from google.colab import files
    uploaded = files.upload()
    dataset_path = [n for n in uploaded if n.lower().endswith(".jsonl")][0]
except Exception:
    dataset_path = "varied_data_generation/dataset.jsonl"
with open(dataset_path, "r", encoding="utf-8") as f:
    raw_data = [json.loads(line) for line in f if line.strip()]
# Same split logic as scripts/dialect_utils.py used by BART no-fine-tune zero-shot flow
def split_counts(total, fractions):
    if total <= 0:
        return (0, 0, 0)
    if not math.isclose(sum(fractions), 1.0, rel_tol=1e-9, abs_tol=1e-9):
        raise ValueError(f"Fractions must sum to 1.0, got {fractions}")
    raw = [total * f for f in fractions]
    counts = [int(math.floor(v)) for v in raw]
    remainder = total - sum(counts)
    order = sorted(range(len(raw)), key=lambda i: (raw[i] - counts[i]), reverse=True)
    for i in order[:remainder]:
        counts[i] += 1
    active = [i for i, f in enumerate(fractions) if f > 0]
    if total < len(active):
        raise ValueError("Not enough items for non-empty requested splits")
    for i in active:
        if counts[i] > 0:
            continue
        donor = max((j for j in active if counts[j] > 1), key=lambda j: counts[j], default=None)
        if donor is None:
            raise ValueError("Unable to keep all requested splits non-empty")
        counts[donor] -= 1
        counts[i] += 1
    return tuple(counts)
def stratified_instance_split(data, seed=42, train_fraction=0.8, val_fraction=0.1, test_fraction=0.1):
    by_family = defaultdict(list)
    for item in data:
        by_family[item["family"]].append(item)
    rng = random.Random(seed)
    train_data, val_data, test_data = [], [], []
    for family in sorted(by_family):
        items = list(by_family[family])
        rng.shuffle(items)
        n_train, n_val, n_test = split_counts(len(items), (train_fraction, val_fraction, test_fraction))
        train_data.extend(items[:n_train])
        val_data.extend(items[n_train : n_train + n_val])
        test_data.extend(items[n_train + n_val : n_train + n_val + n_test])
    return train_data, val_data, test_data
_, _, test_data = stratified_instance_split(raw_data, seed=SEED, train_fraction=0.8, val_fraction=0.1, test_fraction=0.1)
families = sorted({d["labels"]["behavioral"] for d in raw_data})
ops = sorted({op.lower() for d in raw_data for op in d["labels"].get("operators", [])})
op2id = {op: i for i, op in enumerate(ops)}
structure_by_family = {
    "Heat": {"time_order": 1, "first_spatial": 0, "second_spatial": 1, "nonlinear": 0, "spatial_vars": 1},
    "Wave": {"time_order": 2, "first_spatial": 0, "second_spatial": 1, "nonlinear": 0, "spatial_vars": 1},
    "Burgers": {"time_order": 1, "first_spatial": 1, "second_spatial": 1, "nonlinear": 1, "spatial_vars": 1},
    "Laplace": {"time_order": 0, "first_spatial": 0, "second_spatial": 1, "nonlinear": 0, "spatial_vars": 2},
    "Advection": {"time_order": 1, "first_spatial": 1, "second_spatial": 0, "nonlinear": 0, "spatial_vars": 1},
    "KleinGordon": {"time_order": 2, "first_spatial": 0, "second_spatial": 1, "nonlinear": 0, "spatial_vars": 1},
    "ReactionDiffusion": {"time_order": 1, "first_spatial": 0, "second_spatial": 1, "nonlinear": 1, "spatial_vars": 1},
    "Beam": {"time_order": 2, "first_spatial": 0, "second_spatial": 0, "nonlinear": 0, "spatial_vars": 1},
}
tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])
model = AutoModelForSeq2SeqLM.from_pretrained(CONFIG["model_name"]).to(device)
model.eval()
def build_prompt(text):
    return (
        "Return one line exactly in format: "
        "family: <family> | operators: <comma-separated ops> | time_order: <0|1|2> | "
        "first_spatial: <0|1> | second_spatial: <0|1> | nonlinear: <0|1> | spatial_vars: <1|2>\n"
        f"Allowed families: {', '.join(families)}\n"
        f"Allowed operators: {', '.join(ops)}\n"
        f"PDE: {text}"
    )
def parse_output(out):
    fam = re.search(r"family\s*:\s*([^|\n]+)", out, flags=re.I)
    ops_m = re.search(r"operators\s*:\s*([^|\n]+)", out, flags=re.I)
    t = re.search(r"time_order\s*:\s*([0-2])", out, flags=re.I)
    f = re.search(r"first_spatial\s*:\s*([01])", out, flags=re.I)
    s = re.search(r"second_spatial\s*:\s*([01])", out, flags=re.I)
    n = re.search(r"nonlinear\s*:\s*([01])", out, flags=re.I)
    v = re.search(r"spatial_vars\s*:\s*([12])", out, flags=re.I)
    return {
        "family": fam.group(1).strip() if fam else None,
        "ops": [x.strip().lower() for x in ops_m.group(1).split(",") if x.strip()] if ops_m else [],
        "time_order": int(t.group(1)) if t else None,
        "first_spatial": int(f.group(1)) if f else None,
        "second_spatial": int(s.group(1)) if s else None,
        "nonlinear": int(n.group(1)) if n else None,
        "spatial_vars": int(v.group(1)) if v else None,
    }
def micro_f1(pred_rows, gold_rows):
    tp = fp = fn = 0
    for pr, gr in zip(pred_rows, gold_rows):
        for p, g in zip(pr, gr):
            tp += int(p == 1 and g == 1)
            fp += int(p == 1 and g == 0)
            fn += int(p == 0 and g == 1)
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec = tp / (tp + fn) if (tp + fn) else 0.0
    return (2 * prec * rec / (prec + rec)) if (prec + rec) else 0.0
def eval_dialect(dialect):
    fam_ok = t_ok = f_ok = s_ok = n_ok = v_ok = 0
    pred_ops_rows, gold_ops_rows = [], []
    for i in tqdm(range(0, len(test_data), CONFIG["batch_size"]), desc=f"Eval {dialect}"):
        batch = test_data[i : i + CONFIG["batch_size"]]
        enc = tokenizer([build_prompt(x["dialects"][dialect]) for x in batch], max_length=CONFIG["max_input_length"], truncation=True, padding=True, return_tensors="pt").to(device)
        with torch.no_grad():
            out_ids = model.generate(**enc, max_new_tokens=CONFIG["max_new_tokens"], num_beams=1, do_sample=False)
        outs = tokenizer.batch_decode(out_ids, skip_special_tokens=True)
        for gold, out in zip(batch, outs):
            pred = parse_output(out)
            fam = gold["labels"]["behavioral"]
            fam_ok += int(pred["family"] is not None and pred["family"].lower() == fam.lower())
            g = [0] * len(ops)
            for op in gold["labels"].get("operators", []):
                lo = op.lower()
                if lo in op2id:
                    g[op2id[lo]] = 1
            p = [0] * len(ops)
            for op in pred["ops"]:
                if op in op2id:
                    p[op2id[op]] = 1
            gold_ops_rows.append(g)
            pred_ops_rows.append(p)
            target = structure_by_family[fam]
            t_ok += int(pred["time_order"] == target["time_order"])
            f_ok += int(pred["first_spatial"] == target["first_spatial"])
            s_ok += int(pred["second_spatial"] == target["second_spatial"])
            n_ok += int(pred["nonlinear"] == target["nonlinear"])
            v_ok += int(pred["spatial_vars"] == target["spatial_vars"])
    n = len(test_data)
    structure_acc = ((t_ok / n) + (f_ok / n) + (s_ok / n) + (n_ok / n) + (v_ok / n)) / 5.0
    return {"Family Acc": fam_ok / n, "Operator F1": micro_f1(pred_ops_rows, gold_ops_rows), "Structure Acc": structure_acc}
rows = []
for d in DIALECTS:
    m = eval_dialect(d)
    m["Dialect"] = d
    rows.append(m)
result_df = pd.DataFrame(rows).set_index("Dialect")[["Family Acc", "Operator F1", "Structure Acc"]]
display(result_df.style.format("{:.2%}"))

Saving dataset.jsonl to dataset.jsonl


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Eval natural:   0%|          | 0/125 [00:00<?, ?it/s]

Eval latex:   0%|          | 0/125 [00:00<?, ?it/s]

Eval prefix:   0%|          | 0/125 [00:00<?, ?it/s]

Eval postfix:   0%|          | 0/125 [00:00<?, ?it/s]

,Family Acc,Operator F1,Structure Acc
Dialect,,,
natural,0.00%,0.61%,31.36%
latex,0.00%,64.60%,24.48%
prefix,0.00%,0.00%,36.00%
postfix,0.00%,0.00%,36.00%
